# FCVG101 — Tìm sản phẩm thời trang bằng ảnh (DeepFashion In-shop)

**Một notebook cho cả Task 1 (xử lý dữ liệu, biểu diễn ảnh) và Task 2 (mô hình, kết quả, demo, phân tích lỗi).**
Task 3 (hiệu chỉnh camera, lọc/FFT/gradient/điểm đặc trưng, optical flow) là thí nghiệm độc lập, nằm ở notebook riêng.

```text
 Dữ liệu ─► Tiền xử lý ─► CLIP ViT-B/32 (đóng băng) ─► Adapter 512→128→512 ─► vector 512 ─► cosine với gallery ─► Top-K
 (Phần A)   (Phần A)        (Phần B)                      (học bằng ArcFace)                   (Phần B: bảng, demo, lỗi)
```

| Phần | Nội dung | Ô |
|---|---|---|
| 0 | Cấu hình môi trường: Colab hoặc máy cá nhân, đường dẫn dữ liệu, checkpoint | 0 |
| A | Task 1: dataset, chia train/val/test, pipeline, augmentation, biểu diễn ảnh, các cấu hình | `# 2` … `# 10` |
| B | Task 2: bảng kết quả test, demo trên ảnh chưa thấy, phân tích lỗi | `# T2-1` … `# T2-6` |
| Phụ lục | Chạy lại toàn bộ từ đầu (mã hóa, train, đánh giá) — mặc định tắt | cuối |

**Hai chế độ:** mặc định chạy nhanh — dùng checkpoint có sẵn, chỉ mã hóa 12.612 ảnh gallery nếu chưa có.
Muốn tái tạo mọi thứ (vài giờ) thì đặt `RUN_FULL = True` ở ô 0.

**Dữ liệu:** ảnh DeepFashion không được phân phối lại (chỉ dùng nghiên cứu/giáo dục phi thương mại). Cần thư mục có
`img/`, `Eval/list_eval_partition.txt`, `list_bbox_inshop.txt` — tải từ
[trang CUHK](https://mmlab.ie.cuhk.edu.hk/projects/DeepFashion/InShopRetrieval.html).
Trên Colab: đưa thư mục (hoặc `dataset.zip` chứa ba mục trên) vào Google Drive rồi sửa `DRIVE_DATA_DIR`.


In [ ]:
# 0. Cấu hình môi trường — Colab hoặc máy cá nhân
import os, subprocess, sys, zipfile
from pathlib import Path

# ---- chỉnh ở đây -------------------------------------------------------------------------------
RUN_FULL = False                                        # True: chạy lại mã hóa + train + test ở phụ lục (vài giờ)
DATA_DIR = None                                         # máy cá nhân: thư mục chứa img/, Eval/, list_bbox_inshop.txt (None = thư mục cha của repo)
DRIVE_DATA_DIR = "/content/drive/MyDrive/FCVG101_data"  # Colab: thư mục trên Drive (img/ + Eval/ + bbox, hoặc dataset.zip)
REPO_URL = "https://github.com/trungledang/fcvg101-visual-product-search"
CKPT_DRIVE_ID = "1XWptbtHuBulXwOthm_7qVEHB8puaAw7c"     # 9 adapter đã train (4,4 MB), xem MODEL_LINKS.md
# -------------------------------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/repo")
    if not (ROOT / "vps").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ftfy", "regex", "ImageHash", "gdown",
                        "git+https://github.com/openai/CLIP.git"], check=True)
    DATA = Path(DRIVE_DATA_DIR)
    if (DATA / "dataset.zip").exists() and not Path("/content/data/img").is_dir():   # giải nén ra đĩa cục bộ cho nhanh
        zipfile.ZipFile(DATA / "dataset.zip").extractall("/content/data")
    if Path("/content/data/img").is_dir():
        DATA = Path("/content/data")
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "vps").is_dir():
        ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT / "visual-product-search-pilot"
    DATA = Path(DATA_DIR) if DATA_DIR else ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

need = [DATA / "Eval/list_eval_partition.txt", DATA / "list_bbox_inshop.txt", DATA / "img"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"Thiếu dữ liệu DeepFashion: {missing}. Xem hướng dẫn ở ô markdown đầu notebook."

# Checkpoint adapter: dùng bản có sẵn, nếu không có thì tải từ Drive
if not (ROOT / "cache/adapter/ep40/e3_ratio0.2_seed0.pt").exists():
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    (ROOT / "cache/adapter").mkdir(parents=True, exist_ok=True)
    gdown.download(id=CKPT_DRIVE_ID, output=str(ROOT / "cache/adapter/ckpt.zip"), quiet=True)
    zipfile.ZipFile(ROOT / "cache/adapter/ckpt.zip").extractall(ROOT / "cache/adapter")

import torch
from vps.embed import load_model
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_pre, clip_meta = load_model(ROOT / ".cache/clip", device)   # tự tải CLIP chính thức, kiểm tra SHA-256
print("môi trường:", "Colab" if IN_COLAB else "máy cá nhân", "| thiết bị:", device, "| dữ liệu:", DATA)


---
# Phần A — Task 1: xử lý dữ liệu và biểu diễn ảnh


# Task 1 — Xử lý dữ liệu ảnh và biểu diễn ảnh (DeepFashion In-shop)

**Bài toán:** tìm đúng sản phẩm (*item_id*) trong kho ảnh shop từ một ảnh query — image-to-image retrieval.
Notebook này làm phần **Task 1 (30 điểm)** và chuẩn bị cấu hình cho Task 2:

| Yêu cầu đề bài | Ô |
|---|---|
| Hiểu dataset: đặc điểm, định dạng nhãn, phân bố, vấn đề chất lượng | 2–4 |
| Chiến lược train / validation / test, chống leakage | 5 |
| Pipeline: load → crop → pad → resize → normalize → batch | 6–7 |
| Augmentation (chỉ dùng khi train) | 8 |
| Biểu diễn ảnh: handcrafted (HSV histogram) và deep (CLIP) | 9 |
| Baseline + các cấu hình thay thế cho Task 2 | 10 |

**Cách chạy:** kernel `.venv/Scripts/python.exe` của dự án, chạy từ trên xuống.
Mọi ảnh minh hoạ lấy từ **split train**; official query/gallery chỉ được đếm thống kê, không dùng để chọn cấu hình.
Ô 9 nạp CLIP ViT-B/32 (đã có ở `.cache/clip/`) và encode vài chục ảnh — chạy được trên CPU, nhanh hơn trên GPU.


In [ ]:
# 2. Đọc partition chính thức và bbox
%matplotlib inline
from pathlib import Path
from collections import Counter
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# ROOT, DATA, device, clip_model, clip_pre do ô cấu hình (ô 0) đã tạo
from vps.annotations import parse_partition, split_summary, validate_splits
from vps.preprocess import (CLOTHES_TYPES, CONFIGS, POSE_TYPES, BBoxCropPad, crop_box, hsv_histogram,
                            make_transform, pad_to_square, validation_split)
from vps.preprocess import parse_bbox

records = parse_partition((DATA / "Eval/list_eval_partition.txt").read_text())
boxes = parse_bbox((DATA / "list_bbox_inshop.txt").read_text())
assert set(boxes) == {r.path for r in records}, "bbox và partition phải cùng tập ảnh"
SPLITS = ("train", "query", "gallery")
by_split = {s: [r for r in records if r.split == s] for s in SPLITS}

print(f"{len(records):,} ảnh, {len({r.item_id for r in records}):,} sản phẩm, {len({r.category for r in records})} category")
for s, v in split_summary(records).items():
    print(f"  {s:8s}: {v['images']:6,} ảnh | {v['unique_items']:5,} item_id")
with Image.open(DATA / records[0].path) as im:
    print("Ví dụ:", records[0].path, im.size, im.mode, "| bbox:", boxes[records[0].path])


### 3. Định dạng nhãn và phân bố

- `list_eval_partition.txt`: `image_name item_id evaluation_status` — split là `train`, `query` hoặc `gallery`.
- `list_bbox_inshop.txt`: `image_name clothes_type pose_type x_1 y_1 x_2 y_2` trên ảnh 256×256.
  `clothes_type`: 1 thân trên, 2 thân dưới, 3 toàn thân. `pose_type`: 1 trước, 2 nghiêng, 3 sau, 4 zoom-out, 5 zoom-in, 6 chỉ sản phẩm.
- Đây là bài toán **retrieval**, không phải phân loại: nhãn là *item_id*, category chỉ dùng để phân tích.


In [ ]:
# 3. Phân bố category, số ảnh mỗi sản phẩm, pose theo split
cats = sorted({r.category for r in records})
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
bottom = np.zeros(len(cats))
for s in SPLITS:
    n = Counter(r.category for r in by_split[s])
    vals = np.array([n[c] for c in cats])
    axes[0].barh(cats, vals, left=bottom, label=s)
    bottom += vals
axes[0].set(title="Số ảnh theo category và split", xlabel="ảnh")
axes[0].legend(); axes[0].tick_params(axis="y", labelsize=8); axes[0].invert_yaxis()

per_id = np.array(list(Counter(r.item_id for r in records).values()))
axes[1].hist(per_id, bins=np.arange(1, 40), color="gray")
axes[1].set(title=f"Số ảnh mỗi item_id (trung vị {np.median(per_id):.0f}, max {per_id.max()})",
            xlabel="ảnh / sản phẩm", ylabel="số sản phẩm")

poses = list(POSE_TYPES)
x = np.arange(len(poses))
for k, s in enumerate(SPLITS):
    n = Counter(boxes[r.path].pose_type for r in by_split[s])
    axes[2].bar(x + (k - 1) * 0.27, [100 * n[p] / len(by_split[s]) for p in poses], 0.27, label=s)
axes[2].set_xticks(x, [POSE_TYPES[p] for p in poses])
axes[2].set(title="Tỉ lệ pose theo split (%)", ylabel="%"); axes[2].legend()
fig.tight_layout(); plt.show()

zoom = {s: sum(boxes[r.path].pose_type == "5" for r in by_split[s]) / len(by_split[s]) for s in SPLITS}
print("Tỉ lệ zoom-in:", {s: f"{v:.1%}" for s, v in zoom.items()})
print("Mất cân bằng category: lớn nhất/nhỏ nhất =",
      max(Counter(r.category for r in records).values()), "/", min(Counter(r.category for r in records).values()))


**Nhận xét cần đưa vào báo cáo:** category rất mất cân bằng; mỗi sản phẩm chỉ có vài ảnh (bài toán few-shot theo item).
Query có tỉ lệ ảnh **zoom-in** cao hơn hẳn gallery — query thường cận cảnh, gallery thường toàn thân —
đây là một lý do để thử crop theo bbox, giúp hai phía cùng tập trung vào sản phẩm.


In [ ]:
# 4. Chất lượng bbox: kích thước, tỉ lệ khung, box quá hẹp
b = np.array([[v.x1, v.y1, v.x2, v.y2] for v in boxes.values()])
w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].hist(w, bins=64, alpha=0.7, label="rộng"); axes[0].hist(h, bins=64, alpha=0.7, label="cao")
axes[0].set(title="Kích thước bbox (px, ảnh 256×256)"); axes[0].legend()
axes[1].hist(w / h, bins=80, color="gray"); axes[1].axvline(1, color="red", ls="--")
axes[1].set(title=f"Tỉ lệ rộng/cao (trung vị {np.median(w / h):.2f})")
axes[2].hist(w * h / 256 ** 2, bins=50, color="gray")
axes[2].set(title="Diện tích bbox / diện tích ảnh")
fig.tight_layout(); plt.show()
narrow = w < 32
print(f"Bbox rộng < 32 px: {narrow.sum():,} ({narrow.mean():.1%}); hẹp nhất {w.min()} px. Cao < 32 px: {(h < 32).sum()}")
print("Pose của box hẹp:", {POSE_TYPES[k]: v for k, v in Counter(v.pose_type for v, n in zip(boxes.values(), narrow) if n).most_common()})

# Ảnh minh hoạ chọn cố định từ train (hash đường dẫn) để chạy lại ra cùng kết quả
import hashlib
def pick(pred, n):
    rs = [r for r in by_split["train"] if pred(r)]
    return sorted(rs, key=lambda r: hashlib.sha256(r.path.encode()).hexdigest())[:n]

def show_boxes(rs, title):
    fig, axes = plt.subplots(1, len(rs), figsize=(2.6 * len(rs), 3))
    for ax, r in zip(axes, rs):
        bb = boxes[r.path]
        with Image.open(DATA / r.path) as im:
            im = im.convert("RGB"); d = ImageDraw.Draw(im)
            d.rectangle([bb.x1, bb.y1, bb.x2, bb.y2], outline=(255, 0, 0), width=2)
            d.rectangle(crop_box(bb, *im.size), outline=(0, 160, 0), width=2)
        ax.imshow(im); ax.axis("off")
        ax.set_title(f"{POSE_TYPES[bb.pose_type]}, rộng {bb.width}px", fontsize=8)
    fig.suptitle(title, fontsize=10); fig.tight_layout(); plt.show()

show_boxes(pick(lambda r: boxes[r.path].width < 20, 6),
           "Box quá hẹp — đỏ: bbox gốc; xanh: sau khi nới margin 10% và cạnh tối thiểu 64px")


**Vấn đề chất lượng dữ liệu quan sát được:**

- Phần lớn box hẹp nằm ở ảnh **chụp nghiêng**: box chỉ khoanh một dải mảnh dọc mép trang phục, không bao trọn sản phẩm — nhiễu nhãn, không phải lỗi đọc file.
  Pipeline xử lý bằng margin + cạnh tối thiểu 64 px; phương án khác (dùng toàn ảnh khi box hẹp hơn ngưỡng) là tham số để chọn trên **val**.
- Ảnh đều 256×256 JPEG, không thiếu/hỏng (audit của pilot đã giải mã toàn bộ 52.712 ảnh).
- Một `item_id` có thể gồm nhiều màu của cùng mẫu, nên nhãn không tương đương một SKU màu duy nhất.
- Ảnh studio nền sáng, người mẫu chiếm phần lớn ảnh: model có thể dựa vào người mẫu/tư thế thay vì sản phẩm.


In [ ]:
# 5. Chiến lược split và kiểm tra leakage
facts = validate_splits(records)          # raise nếu query/gallery trùng ảnh hoặc train/test trùng ID
val = validation_split(records)           # tách 10% item_id của train làm validation
for k, v in facts.items():
    if not k.endswith("examples"):
        print(f"  {k}: {v}")
ids = {k: {r.item_id for r in v} for k, v in val.items()}
test_ids = {r.item_id for r in by_split["query"] + by_split["gallery"]}
assert not ids["train"] & ids["val_query"], "train và val không chung sản phẩm"
assert not (ids["train"] | ids["val_query"]) & test_ids, "không chạm vào item của official test"
assert ids["val_query"] == ids["val_gallery"], "mỗi item val có cả ảnh query và gallery"
assert not {r.path for r in val["val_query"]} & {r.path for r in val["val_gallery"]}
print()
for k, v in val.items():
    print(f"  {k:12s}: {len(v):6,} ảnh | {len({r.item_id for r in v}):5,} item_id")


| Tập | Dùng để | Ghi chú |
|---|---|---|
| **train** (90% item_id của split train) | Học/adapt ở Task 2 | Augmentation chỉ áp dụng ở đây |
| **val_query / val_gallery** (10% item_id của train) | Chọn cấu hình, margin, epoch, early stopping | Chia theo *sản phẩm*, cùng kiểu query→gallery như test |
| **official query / gallery** | Đánh giá cuối, **một lần** cho mỗi cấu hình đã chốt | Giữ toàn bộ 12.612 ảnh gallery làm distractor |

Chống leakage: (1) chia theo *item_id*, không chia theo ảnh — ảnh cùng sản phẩm không lọt giữa train và val/test;
(2) official split đảm bảo train và test không chung sản phẩm, query và gallery không chung ảnh (đã assert ở trên);
(3) audit của pilot không thấy ảnh trùng byte/pixel giữa các split; pHash tìm thấy cặp gần giống nhưng không tự loại;
(4) không chọn tham số trên test. Giới hạn: không loại trừ được việc CLIP từng thấy ảnh DeepFashion khi pretrain.


In [ ]:
# 6. Pipeline từng bước trên ảnh train đại diện cho mỗi pose
import torch, clip
from torchvision import transforms as T
preprocess = clip_pre
print("CLIP preprocess:", preprocess)
MEAN = torch.tensor(preprocess.transforms[-1].mean).view(3, 1, 1)
STD = torch.tensor(preprocess.transforms[-1].std).view(3, 1, 1)
def visible(t):  # hoàn tác normalize CHỈ để hiển thị
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

c0 = make_transform("C0_full", preprocess)
c1 = make_transform("C1_bbox_pad", preprocess, boxes)
def naive_crop(im, path):  # crop bbox rồi để CLIP center-crop luôn: phản ví dụ
    return preprocess(im.convert("RGB").crop(crop_box(boxes[path], *im.size)))

samples = [pick(lambda r, p=p: boxes[r.path].pose_type == p, 1)[0] for p in ("1", "2", "4", "5")]
cols = ["Ảnh gốc + bbox", "C0: toàn ảnh (baseline)", "Crop bbox → center crop\n(mất đầu/cuối)", "C1: crop bbox → pad vuông"]
fig, axes = plt.subplots(len(samples), 4, figsize=(11, 2.9 * len(samples)))
for row, r in zip(axes, samples):
    bb = boxes[r.path]
    with Image.open(DATA / r.path) as im:
        im = im.convert("RGB")
        shown = im.copy(); ImageDraw.Draw(shown).rectangle([bb.x1, bb.y1, bb.x2, bb.y2], outline=(255, 0, 0), width=2)
        views = [np.asarray(shown), visible(c0(im, r.path)), visible(naive_crop(im, r.path)), visible(c1(im, r.path))]
    for ax, v, title in zip(row, views, cols):
        ax.imshow(v); ax.axis("off"); ax.set_title(title, fontsize=8)
    row[0].set_title(f"{cols[0]}\n{POSE_TYPES[bb.pose_type]} · {CLOTHES_TYPES[bb.clothes_type]}", fontsize=8)
fig.tight_layout(); plt.show()


**Các bước pipeline và lý do:**

1. **Load + convert RGB** (PIL): đảm bảo 3 kênh dù file có mode khác.
2. **Crop theo bbox** (C1, C2): nới **10% mỗi cạnh** để giữ viền/cổ áo, nới cạnh tối thiểu **64 px** cho box nghiêng quá hẹp, rồi dịch box vào trong ảnh nếu tràn biên.
3. **Pad về hình vuông** bằng màu trung bình của CLIP: bbox thường cao hơn rộng, nếu đưa thẳng vào `Resize(224) + CenterCrop(224)` của CLIP sẽ cắt mất phần trên/dưới (cột 3).
   Màu pad bằng mean nên sau normalize gần bằng 0; nhưng nó khác nền trắng của studio, nên màu pad (mean / trắng) là tham số so trên val.
   **Kết quả val** (`reports/val_crop_ablation.json`): pad trắng 65,8% R@1 so với pad mean 53,6%; ảnh đầy đủ (C0) vẫn cao nhất 71,1% → Task 2 giữ C0.
4. **Resize 224 bicubic + center crop 224**: giữ đúng preprocessing CLIP đã học khi pretrain; với ảnh đã vuông bước crop không bỏ mất gì.
5. **ToTensor + Normalize** theo mean/std của CLIP.
6. **Batch** bằng `DataLoader` (ô 7). Embedding đầu ra được L2-normalize để cosine = tích vô hướng.


In [ ]:
# 7. Batching: Dataset trả (tensor, item_id); DataLoader gom thành batch
class InShop(torch.utils.data.Dataset):
    def __init__(self, recs, transform):
        self.recs, self.transform = recs, transform
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        with Image.open(DATA / r.path) as im:
            return self.transform(im, r.path), r.item_id

loader = torch.utils.data.DataLoader(InShop(val["val_query"][:16], c1), batch_size=8, shuffle=False)
x, item_ids = next(iter(loader))
print("batch:", tuple(x.shape), x.dtype, "| item_id đầu:", item_ids[:3])
print(f"sau normalize: mean {x.mean():.3f}, std {x.std():.3f}, min {x.min():.2f}, max {x.max():.2f}")


In [ ]:
# 8. Augmentation cho train (C2) — eval vẫn dùng transform xác định của C1
c2_train = make_transform("C2_bbox_pad_aug", preprocess, boxes, train=True)
r = samples[0]
torch.manual_seed(0)
with Image.open(DATA / r.path) as im:
    im.load()
    views = [visible(c1(im, r.path))] + [visible(c2_train(im, r.path)) for _ in range(7)]
fig, axes = plt.subplots(1, 8, figsize=(18, 2.8))
for k, (ax, v) in enumerate(zip(axes, views)):
    ax.imshow(v); ax.axis("off"); ax.set_title("C1 (eval)" if k == 0 else f"C2 train #{k}", fontsize=8)
fig.tight_layout(); plt.show()


| Augmentation | Tham số | Lý do |
|---|---|---|
| RandomResizedCrop | scale 0.75–1, ratio 0.9–1.1 | Mô phỏng zoom/lệch khung giữa query và gallery |
| HorizontalFlip | p = 0.5 | Quần áo gần như đối xứng trái–phải |
| ColorJitter | **chỉ** brightness/contrast 0.2 | Ánh sáng studio thay đổi; **không** đổi hue/saturation vì màu là đặc trưng nhận diện sản phẩm |
| RandomErasing | p = 0.25, 2–10% diện tích | Mô phỏng che khuất bởi tay/tóc/phụ kiện |

Không dùng xoay lớn hay lật dọc: ảnh shop luôn thẳng đứng. Augmentation chỉ có tác dụng khi **train** (Task 2);
với model frozen ở baseline, C2 khi đánh giá bằng đúng C1.


In [ ]:
# 9. Biểu diễn ảnh: HSV histogram (handcrafted) so với CLIP embedding (deep), trên val
model = clip_model  # đã nạp ở ô 0, đóng băng, chế độ eval
crop = BBoxCropPad(boxes)

vq = sorted(val["val_query"], key=lambda r: hashlib.sha256(r.path.encode()).hexdigest())[:5]
vg = val["val_gallery"]
def load(r):
    with Image.open(DATA / r.path) as im:
        return im.convert("RGB")
@torch.inference_mode()
def clip_emb(recs, transform):
    x = torch.stack([transform(load(r), r.path) for r in recs]).to(device)
    f = model.encode_image(x).float()
    return (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()

gal_ids = np.array([r.item_id for r in vg])
reps = {
    "HSV (crop+pad)": (np.stack([hsv_histogram(crop(load(r), r.path)) for r in vq]),
                       np.stack([hsv_histogram(crop(load(r), r.path)) for r in vg])),
    "CLIP C0 toàn ảnh": (clip_emb(vq, c0), clip_emb(vg, c0)),
    "CLIP C1 crop+pad": (clip_emb(vq, c1), clip_emb(vg, c1)),
}
print(f"{len(vq)} query val × {len(vg):,} ảnh val gallery; hạng của ảnh đúng sản phẩm đầu tiên (1 = tốt nhất):")
for name, (q, g) in reps.items():
    s = q @ g.T
    ranks = [int((s[i] > s[i][gal_ids == r.item_id].max()).sum()) + 1 for i, r in enumerate(vq)]
    print(f"  {name:18s} dim={q.shape[1]:4d}  hạng = {ranks}")

r = vq[0]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].imshow(crop(load(r), r.path)); axes[0].axis("off"); axes[0].set_title(f"Query val {r.item_id}", fontsize=9)
axes[1].bar(range(128), reps["HSV (crop+pad)"][0][0], color="gray")
axes[1].set(title="HSV histogram 8×4×4 = 128 chiều", xlabel="bin")
axes[2].plot(reps["CLIP C1 crop+pad"][0][0], linewidth=0.7)
axes[2].set(title="CLIP embedding 512 chiều", xlabel="chiều")
fig.tight_layout(); plt.show()


Ô 9 chỉ minh hoạ trên **5 query val** để thấy cách hai loại biểu diễn hoạt động — **không phải kết quả thực nghiệm**.
So sánh chính thức làm ở Task 2 trên toàn bộ val, chốt cấu hình, rồi mới chạy test một lần.

- **HSV histogram**: chỉ đếm màu, mất toàn bộ thông tin hình dạng/hoa văn; rẻ, dễ giải thích, là mốc dưới.
- **CLIP embedding**: đặc trưng ngữ nghĩa học từ 400 triệu cặp ảnh–văn bản; đã đo trên test: R@1 45,5% (C0).


In [ ]:
# 10. Các cấu hình đưa sang Task 2
desc = {
    "C0_full": "Baseline: toàn ảnh, preprocessing CLIP gốc",
    "C1_bbox_pad": "Crop bbox (+10%, cạnh ≥ 64px) → pad vuông → CLIP",
    "C2_bbox_pad_aug": "C1 + augmentation khi train (crop/flip/sáng-tương phản/erasing)",
    "C3_hsv_hist": "Handcrafted: HSV histogram 128 chiều trên ảnh crop+pad",
    "C0_full_aug": "Ảnh đầy đủ + augmentation khi train (dùng cho E2 ở Phần B)",
}
print(f"{'Cấu hình':16s} {'crop':5s} {'aug':5s} {'biểu diễn':9s} mô tả")
for k, v in CONFIGS.items():
    print(f"{k:16s} {str(v['crop']):5s} {str(v['train_augment']):5s} {v['representation']:9s} {desc[k]}")


**Thứ tự so sánh ở Task 2** (đổi một yếu tố mỗi lần, cùng split và cùng Recall@1/5):

1. C0 → C1 với CLIP frozen: tác động của **crop/pad** (chỉ đổi preprocessing).
2. C0 / C1 → C3: **deep vs handcrafted** cùng crop.
3. C1 → C2 khi **adapt** CLIP (ví dụ projection head hoặc fine-tune vài block cuối, train trên `train`, chọn epoch trên val): tác động của **augmentation**.

Chọn tham số (margin, cạnh tối thiểu, số epoch, learning rate) bằng **val**; official test chỉ chạy một lần cho mỗi cấu hình đã chốt.


---


# Phần B — Task 2: kết quả, demo suy luận và phân tích lỗi

**Hệ thống:** ảnh → CLIP ViT-B/32 (đóng băng) → adapter 512→128→512 (residual, ratio 0.2, học bằng ArcFace) → so cosine với 12.612 ảnh gallery → Top-K.

| Yêu cầu Task 2 | Ô |
|---|---|
| Bảng so sánh trên test + hình | T2-2 |
| Demo suy luận trên mẫu chưa thấy | T2-3, T2-4 |
| Phân tích lỗi ngắn | T2-5, T2-6 |

**Nguyên tắc:** cấu hình đã chốt trên validation trước khi chạy test (`docs/T1_ADAPTER_GIAI_THICH.md`, mục 5c).
Notebook này chỉ đọc lại số test (`reports/test_results.json`), không chọn gì dựa trên test.
Mọi query test thuộc 3.985 sản phẩm **không có trong tập train** → mỗi query là một mẫu chưa thấy.

**Dữ liệu cần có:** cấu hình ở ô 0 (đầu notebook). Chạy được trên CPU, nhanh hơn trên GPU.


In [ ]:
# T2-1. Dựng 2 bộ tìm kiếm: E0 (CLIP gốc) và E3 (adapter, crop + pad trắng)
# Lần đầu nếu chưa có vector gallery thì tự mã hóa 12.612 ảnh (vài phút trên GPU) và lưu vào cache/embeddings/.
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from vps.search import ProductSearch
from vps.annotations import parse_partition

search = {name: ProductSearch(ROOT, name, clip_model, clip_pre, seed=0, data_root=DATA, device=device)
          for name in ("E0_full", "E3")}
records = parse_partition((DATA / "Eval/list_eval_partition.txt").read_text())
queries = [r for r in records if r.split == "query"]
train_ids = {r.item_id for r in records if r.split == "train"}
print(device, "| gallery:", len(search["E3"].gallery), "ảnh | query test:", len(queries))


In [ ]:
# T2-2. Kết quả test (một lần duy nhất) + R@1 theo nhóm sản phẩm
res = json.loads((ROOT / "reports/test_results.json").read_text(encoding="utf-8"))
names = {"E0_full": "E0 CLIP gốc, toàn ảnh", "E0_crop": "E0 CLIP gốc, crop+pad trắng",
         "E1": "E1 + adapter", "E2": "E2 + adapter + augmentation", "E3": "E3 + adapter, crop+pad trắng"}
print(f"{'Cấu hình':32s} {'R@1':>7s} {'R@5':>7s} {'R@10':>7s} {'MRR':>7s}   (adapter: trung bình 3 seed)")
for k, v in res["results"].items():
    a = v["aggregate"]
    print(f"{names[k]:32s} " + " ".join(f"{a[m]['mean']*100:7.2f}" for m in ("R@1", "R@5", "R@10", "MRR")))

cat = res["R@1_by_category_seed0"]
order = sorted(cat, key=lambda c: cat[c]["E0_full"])
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(y - 0.2, [cat[c]["E0_full"] * 100 for c in order], 0.4, label="E0 CLIP gốc", color="#9aa5b1")
ax.barh(y + 0.2, [cat[c]["E3"] * 100 for c in order], 0.4, label="E3 adapter", color="#2f6fb0")
ax.set_yticks(y, [f"{c} ({cat[c]['queries']})" for c in order], fontsize=8)
ax.set_xlabel("R@1 trên test (%)"); ax.legend(loc="lower right"); ax.grid(axis="x", alpha=0.3)
ax.set_title("R@1 theo nhóm sản phẩm (số query trong ngoặc), seed 0")
plt.tight_layout(); plt.show()


**Đọc bảng:** adapter tăng R@1 từ 45,5% lên 71–72% trên cả 23 nhóm. Crop làm CLIP gốc *kém đi* (39,6%)
nhưng lại nhỉnh nhất khi có adapter (E3). E2 (thêm augmentation) không hơn E1 — augmentation nhẹ gần như không đổi
embedding của CLIP đóng băng (cosine giữa các bản ≈ 0,93), nên adapter không học thêm được gì.


In [ ]:
# T2-3. Demo suy luận: query test (sản phẩm chưa từng thấy khi train) → Top-5, E0 và E3 cạnh nhau
def show_row(axes, img, title, hits, true_id):
    axes[0].imshow(img); axes[0].set_title(title, fontsize=8)
    for ax, h in zip(axes[1:], hits):
        ok = h.item_id == true_id
        with Image.open(DATA / h.path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"#{h.rank} {'ĐÚNG' if ok else 'sai'}\ncos {h.score:.3f}", fontsize=8,
                     color="green" if ok else "red")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])

def compare(q, k=5):
    assert q.item_id not in train_ids  # đúng là sản phẩm chưa thấy
    fig, axes = plt.subplots(2, k + 1, figsize=(2.1 * (k + 1), 5.2))
    with Image.open(DATA / q.path) as im:
        for row, name in enumerate(("E0_full", "E3")):
            s = search[name]
            shown, how = s.prepare(im, q.path)
            show_row(axes[row], shown, f"{name}\n{how}", s.search(im, q.path, k=k), q.item_id)
    fig.suptitle(f"Query {q.path}  (item {q.item_id}, {q.category})", fontsize=9)
    plt.tight_layout(); plt.show()

rng = np.random.default_rng(2026)          # chọn ngẫu nhiên có seed, không chọn tay
for i in rng.choice(len(queries), 3, replace=False):
    compare(queries[int(i)])


**Cách đọc:** hàng trên là CLIP gốc, hàng dưới là adapter. Viền chữ xanh = cùng `item_id` với query.
Các mẫu ở trên được chọn ngẫu nhiên có seed — có thể cả hai cùng đúng hoặc cùng sai; đó là hình ảnh thật của hệ thống,
không phải ví dụ được chọn cho đẹp. Muốn xem ví dụ khác: đổi seed `2026`.


In [ ]:
# T2-4. Ảnh bất kỳ (ví dụ ảnh tự chụp): đặt đường dẫn vào MY_IMAGE.
# Ảnh ngoài dataset không có bbox → E3 chỉ pad trắng, không crop (đầu ra ghi rõ điều này).
MY_IMAGE = None   # ví dụ: r"E:\Downloads\ao_cua_toi.jpg"
path = Path(MY_IMAGE) if MY_IMAGE else DATA / queries[0].path   # mặc định: 1 ảnh query, coi như ảnh lạ
fig, axes = plt.subplots(2, 6, figsize=(12.6, 5.2))
with Image.open(path) as im:
    for row, name in enumerate(("E0_full", "E3")):
        s = search[name]
        shown, how = s.prepare(im, None)     # path=None: hệ thống không biết bbox hay item_id
        hits = s.search(im, None, k=5)
        show_row(axes[row], shown, f"{name}\n{how}", hits, true_id=None)
        print(name, [(h.item_id, round(h.score, 3)) for h in hits])
plt.tight_layout(); plt.show()


In [ ]:
# T2-5. Phân tích lỗi: query nào adapter sửa được, query nào làm hỏng (seed 0, toàn bộ 14.218 query)
ranks = np.load(ROOT / "reports/test_first_pos_ranks_E0_E3.npz")  # hạng đã lưu từ scripts/eval_test.py
r0, r3 = ranks["E0_full_seedNone"], ranks["E3_seed0"]
ok0, ok3 = r0 == 1, r3 == 1
print(f"Cả hai đúng     : {np.sum(ok0 & ok3):6d}")
print(f"E3 sửa được     : {np.sum(~ok0 & ok3):6d}   (E0 sai, E3 đúng)")
print(f"E3 làm hỏng     : {np.sum(ok0 & ~ok3):6d}   (E0 đúng, E3 sai)")
print(f"Cả hai sai      : {np.sum(~ok0 & ~ok3):6d}")
fail = r3[~ok3]
print(f"Khi E3 sai ở #1: món đúng nằm ở #2–5 trong {np.mean(fail <= 5):.0%} trường hợp, "
      f"#6–10 {np.mean((fail > 5) & (fail <= 10)):.0%}, ngoài top-10 {np.mean(fail > 10):.0%}")
fig, ax = plt.subplots(figsize=(7, 3))
bins = [1, 2, 3, 4, 5, 6, 11, 21, 51, 101, 13000]
lab = ["1", "2", "3", "4", "5", "6–10", "11–20", "21–50", "51–100", ">100"]
for r, n, c in ((r0, "E0", "#9aa5b1"), (r3, "E3", "#2f6fb0")):
    h, _ = np.histogram(r, bins=bins)
    ax.plot(lab, h / len(r) * 100, "o-", label=n, color=c)
ax.set_ylabel("% query"); ax.set_xlabel("hạng của ảnh đúng đầu tiên"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# T2-6. Sáu lỗi của E3 (chọn ngẫu nhiên có seed) để xếp nhóm lỗi — Phong điền bảng ở ô dưới
fail_idx = np.flatnonzero(~ok3)
for i in np.random.default_rng(7).choice(fail_idx, 6, replace=False):
    print(f"query #{i}: E0 hạng {r0[i]}, E3 hạng {r3[i]}")
    compare(queries[int(i)])


**Nhóm lỗi (Phong điền sau khi xem ô 6 và thêm ví dụ nếu cần):**

| Nhóm lỗi | Ví dụ (query #) | Giải thích ngắn | Adapter có giúp không? |
|---|---|---|---|
| Cùng kiểu dáng/màu, khác món (gần như trùng) | | | |
| Query cận cảnh chi tiết, gallery toàn thân | | | |
| Họa tiết/in hình nhỏ (Graphic_Tees) | | | |
| Góc chụp nghiêng / bbox hẹp | | | |
| Khác (ghi rõ) | | | |

Gợi ý: nhóm khó nhất trên test là WOMEN/Graphic_Tees (E0 18,9% → E3 46,0%) — CLIP ở 224×224 khó đọc hình in nhỏ.


---
# Phụ lục — chạy lại toàn bộ từ đầu (mặc định tắt, `RUN_FULL = False`)

Thứ tự đúng với lúc làm thí nghiệm. Mỗi bước ghi vào `cache/` hoặc `reports/`. Mã hóa ảnh lần đầu có thể mất hàng giờ
nếu đọc ảnh từ ổ chậm (6.305 s trên ổ E: của máy gốc, 96 s khi ảnh đã nằm trong bộ nhớ đệm). Train 40 epoch: ~45 s mỗi lần.
**[Các lệnh dưới đây là các lệnh đã dùng thật; riêng việc chạy chúng từ trong notebook này chưa được chạy trọn vẹn.]**


In [ ]:
# Phụ lục. Tái tạo: mã hóa -> train E1/E2/E3 x 3 seed -> đánh giá test một lần
d = ["--data-root", str(DATA)]
steps = [
    ["scripts/pilot.py", *d, "embed", "--batch-size", "64"],                               # query/gallery, ảnh đầy đủ (E0)
    *[["scripts/encode_variant.py", "--variant", "A1_white_pad", "--split", sp, *d] for sp in ("train", "query", "gallery")],
    ["scripts/encode_augmented.py", *d],                                                    # 3 bản augmented / ảnh train (E2)
]
for seed in (0, 1, 2):
    base = ["scripts/train_adapter.py", *d, "--epochs", "40", "--seed", str(seed), "--ckpt-dir", "cache/adapter/ep40"]
    steps += [base,                                                                         # E1
              base + ["--aug-cache", "train_aug_C0_full_aug_v3_s0"],                        # E2
              base + ["--base-cache", "train_A1_white_pad"]]                                # E3
steps.append(["scripts/eval_test.py", *d])                                                   # chạy test MỘT lần
if RUN_FULL:
    for st in steps:
        print(">", " ".join(st))
        subprocess.run([sys.executable, *st], cwd=ROOT, check=True)
else:
    print(f"RUN_FULL = False: bỏ qua {len(steps)} lệnh. Danh sách:")
    for st in steps:
        print("  python", " ".join(st))
